# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |TS007-Age by single year|
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |TS007_Age.xlsx|
| Output File |age.json |
| Notebook Author | Peter |
| Last Updated |2026-07-11|

## 1. Import

In [15]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [16]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [17]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [18]:
# File paths

input_file = "TS007_Age.xlsx"
output_file = "age.json"

raw_data_path = Path("../../data/demographic/raw/age")
output_data_path = Path("../../data/demographic/processed/age")

## 3. Load Dataset

In [19]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,0,Aged under 1 year,918
1,E06000001,Hartlepool,1,Aged 1 year,954
2,E06000001,Hartlepool,2,Aged 2 years,1046
3,E06000001,Hartlepool,3,Aged 3 years,988
4,E06000001,Hartlepool,4,Aged 4 years,1077


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [20]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28466 entries, 0 to 28465
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Lower tier local authorities Code  28466 non-null  object
 1   Lower tier local authorities       28466 non-null  object
 2   Age (86 categories) Code           28466 non-null  int64 
 3   Age (86 categories)                28466 non-null  object
 4   Observation                        28466 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 1.1+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Age (86 categories) Code', 'Age (86 categories)', 'Observation'],
      dtype='object')

In [21]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Age (86 categories) Code,Age (86 categories),Observation
count,28466,28466,28466.000000,28466,28466.000000
unique,331,331,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Aged under 1 year,NaN
freq,86,86,NaN,331,NaN
mean,NaN,NaN,42.500000,NaN,2093.642275
std,NaN,NaN,24.824819,NaN,1629.631845
min,NaN,NaN,0.000000,NaN,4.000000
25%,NaN,NaN,21.000000,NaN,1121.000000
50%,NaN,NaN,42.500000,NaN,1583.000000
75%,NaN,NaN,64.000000,NaN,2544.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [22]:
# Cleaning code

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [23]:
# Dataset-specific processing
from ontology.age_ontology import (
    age_group,
    age_order
)

In [24]:
def get_age_distribution(local_authority):
    processed_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    processed_data['Age Group'] = processed_data['Age (86 categories) Code'].apply(age_group)

    result = (processed_data.groupby(['Lower tier local authorities', 'Age Group'], as_index=False)['Observation'].sum())

    result['Age Group'] = pd.Categorical(result['Age Group'],categories=age_order,ordered=True)

    result = result.sort_values('Age Group')

    result = result.reset_index(drop=True)

    return result

In [25]:
get_age_distribution("Manchester")

,Lower tier local authorities,Age Group,Observation
0,Manchester,0-4,34378
1,Manchester,5-11,51349
2,Manchester,12-17,41226
3,Manchester,18-22,61780
4,Manchester,23-34,123440
5,Manchester,35-49,108897
6,Manchester,50-64,78705
7,Manchester,65-74,30299
8,Manchester,75+,21868


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [26]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28466 entries, 0 to 28465
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Lower tier local authorities Code  28466 non-null  object
 1   Lower tier local authorities       28466 non-null  object
 2   Age (86 categories) Code           28466 non-null  int64 
 3   Age (86 categories)                28466 non-null  object
 4   Observation                        28466 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 1.1+ MB


Lower tier local authorities Code    0
Lower tier local authorities         0
Age (86 categories) Code             0
Age (86 categories)                  0
Observation                          0
dtype: int64

## 8. Export Processed Dataset

In [27]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

age_df = get_age_distribution("Manchester")

age_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

In [28]:
saved_age_df = pd.read_json(output_data_path / output_file)

display(saved_age_df)

,Lower tier local authorities,Age Group,Observation
0,Manchester,0-4,34378
1,Manchester,5-11,51349
2,Manchester,12-17,41226
3,Manchester,18-22,61780
4,Manchester,23-34,123440
5,Manchester,35-49,108897
6,Manchester,50-64,78705
7,Manchester,65-74,30299
8,Manchester,75+,21868


## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported